In [1]:
!pip install folium plotly geopy

In [2]:
import pandas as pd
import plotly.express as px
import folium
from folium.plugins import HeatMap

In [3]:
from google.colab import files

uploaded = files.upload()

Saving 4Sample - Superstore.csv to 4Sample - Superstore (1).csv


In [4]:
df = pd.read_csv("4Sample - Superstore.csv", encoding="latin1")

df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [5]:
print(df.shape)

df.info()

df.describe()

(9994, 21)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null   float64
 1

,Row ID,Postal Code,Sales,Quantity,Discount,Profit
count,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000
mean,4997.500000,55190.379428,229.858001,3.789574,0.156203,28.656896
std,2885.163629,32063.693350,623.245101,2.225110,0.206452,234.260108
min,1.000000,1040.000000,0.444000,1.000000,0.000000,-6599.978000
25%,2499.250000,23223.000000,17.280000,2.000000,0.000000,1.728750
50%,4997.500000,56430.500000,54.490000,3.000000,0.200000,8.666500
75%,7495.750000,90008.000000,209.940000,5.000000,0.200000,29.364000
max,9994.000000,99301.000000,22638.480000,14.000000,0.800000,8399.976000


In [6]:
print(df.isnull().sum())

Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64


In [7]:
state_sales = df.groupby("State")["Sales"].sum().reset_index()

state_sales = state_sales.sort_values(by="Sales", ascending=False)

state_sales.head(10)

,State,Sales
3,California,457687.6315
30,New York,310876.2710
41,Texas,170188.0458
45,Washington,138641.2700
36,Pennsylvania,116511.9140
8,Florida,89473.7080
11,Illinois,80166.1010
33,Ohio,78258.1360
20,Michigan,76269.6140
44,Virginia,70636.7200


In [8]:
fig = px.bar(
    state_sales.head(10),
    x="State",
    y="Sales",
    color="Sales",
    title="Top 10 States by Sales"
)

fig.show()

In [9]:
region_sales = df.groupby("Region")["Sales"].sum().reset_index()

fig = px.pie(
    region_sales,
    names="Region",
    values="Sales",
    title="Sales Distribution by Region"
)

fig.show()

In [10]:
city_sales = df.groupby("City")["Sales"].sum().reset_index()

city_sales = city_sales.sort_values(by="Sales", ascending=False)

city_sales.head(15)

,City,Sales
329,New York City,256368.1610
266,Los Angeles,175851.3410
452,Seattle,119540.7420
438,San Francisco,112669.0920
374,Philadelphia,109077.0130
207,Houston,64504.7604
80,Chicago,48539.5410
437,San Diego,47521.0290
216,Jacksonville,44713.1830
464,Springfield,43054.3420


In [11]:
coordinates = {
"New York":(40.7128,-74.0060),
"Los Angeles":(34.0522,-118.2437),
"Seattle":(47.6062,-122.3321),
"San Francisco":(37.7749,-122.4194),
"Houston":(29.7604,-95.3698),
"Chicago":(41.8781,-87.6298),
"Philadelphia":(39.9526,-75.1652),
"Phoenix":(33.4484,-112.0740),
"San Diego":(32.7157,-117.1611),
"Dallas":(32.7767,-96.7970),
"Columbus":(39.9612,-82.9988),
"Austin":(30.2672,-97.7431),
"Jacksonville":(30.3322,-81.6557),
"Detroit":(42.3314,-83.0458),
"Denver":(39.7392,-104.9903),
"Charlotte":(35.2271,-80.8431),
"Boston":(42.3601,-71.0589),
"Indianapolis":(39.7684,-86.1581),
"Nashville":(36.1627,-86.7816),
"Portland":(45.5152,-122.6784)
}

In [12]:
city_sales["Latitude"] = city_sales["City"].map(lambda x: coordinates.get(x,(None,None))[0])

city_sales["Longitude"] = city_sales["City"].map(lambda x: coordinates.get(x,(None,None))[1])

city_sales = city_sales.dropna()

city_sales.head()

,City,Sales,Latitude,Longitude
266,Los Angeles,175851.3410,34.0522,-118.2437
452,Seattle,119540.7420,47.6062,-122.3321
438,San Francisco,112669.0920,37.7749,-122.4194
374,Philadelphia,109077.0130,39.9526,-75.1652
207,Houston,64504.7604,29.7604,-95.3698


In [13]:
m = folium.Map(location=[39,-98], zoom_start=4)

for _, row in city_sales.iterrows():

    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=row["Sales"]/20000,
        popup=f'{row["City"]}<br>Sales=${row["Sales"]:.0f}',
        color='blue',
        fill=True,
        fill_color='red'
    ).add_to(m)

m

In [14]:
heat_data = city_sales[['Latitude','Longitude','Sales']].values.tolist()

heatmap = folium.Map(location=[39,-98], zoom_start=4)

HeatMap(heat_data).add_to(heatmap)

heatmap

In [15]:
median_sales = city_sales["Sales"].median()

recommend = city_sales[city_sales["Sales"] > median_sales]

recommend = recommend.sort_values(by="Sales", ascending=False)

recommend.head(10)

,City,Sales,Latitude,Longitude
266,Los Angeles,175851.3410,34.0522,-118.2437
452,Seattle,119540.7420,47.6062,-122.3321
438,San Francisco,112669.0920,37.7749,-122.4194
374,Philadelphia,109077.0130,39.9526,-75.1652
207,Houston,64504.7604,29.7604,-95.3698
80,Chicago,48539.5410,41.8781,-87.6298
437,San Diego,47521.0290,32.7157,-117.1611
216,Jacksonville,44713.1830,30.3322,-81.6557
123,Detroit,42446.9440,42.3314,-83.0458


In [16]:
fig = px.scatter_geo(
    recommend,
    lat="Latitude",
    lon="Longitude",
    hover_name="City",
    size="Sales",
    color="Sales",
    title="Recommended New Store Locations"
)

fig.show()

In [17]:
print("Top Recommended Cities for New Stores:\n")

print(recommend[["City","Sales"]].head(10))

Top Recommended Cities for New Stores:

              City        Sales
266    Los Angeles  175851.3410
452        Seattle  119540.7420
438  San Francisco  112669.0920
374   Philadelphia  109077.0130
207        Houston   64504.7604
80         Chicago   48539.5410
437      San Diego   47521.0290
216   Jacksonville   44713.1830
123        Detroit   42446.9440
